# Model D1 Final Build Runtime Contract

Clean thesis-safe notebook for the final D1 6-channel 1D-CNN + BiLSTM path.

Protocol:
1. ADAMSense/anxiety_dataset supervised pretraining.
2. Zero-shot Garmin transfer check using the ADAMSense scaler.
3. Garmin adaptation sanity check: train users 1-5, validate user 6.
4. Mentor-comparable 5-fold window-level CV on Garmin users 1-6.
5. LOUO across Garmin users 1-6 using train-only scaler per fold.
6. Freeze final model on all Garmin users 1-6 and save runtime contract.
7. User 7 single-shot blind probe after freeze only.

No D2/D3/RF/debug experiments live in this notebook. Those belong in `Model_D_Experiment_Archive.ipynb`.


In [5]:
# ============================================================
# Cell 1 ? Setup: GPU, seeds, paths, constants
# ============================================================

import os
import json
import random
import hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import tensorflow as tf
import joblib

from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras import mixed_precision

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

SEED = 942
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# GPU setup
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    mixed_precision.set_global_policy("mixed_float16")
    print("GPU found:", gpus[0].name)
else:
    mixed_precision.set_global_policy("float32")
    print("WARNING: no GPU detected; running on CPU")

tf.config.optimizer.set_jit(False)

ROOT = Path(r"d:/study apply/ML Ops/MasterArbeit_MLops")
assert ROOT.exists(), f"ROOT not found: {ROOT}"

ART = ROOT / "notebooks" / "model_HAR_27may_artifacts"
ART.mkdir(parents=True, exist_ok=True)

SENSOR_COLS = ["Ax_w", "Ay_w", "Az_w", "Gx_w", "Gy_w", "Gz_w"]
WINDOW = 200
STRIDE = 100
N_CLASSES = 11

LABEL_ORDER = [
    "ear_rubbing", "forehead_rubbing", "hair_pulling", "hand_scratching",
    "hand_tapping", "knuckles_cracking", "nail_biting", "nape_rubbing",
    "sitting", "smoking", "standing",
]
LABEL_TO_IDX = {label: i for i, label in enumerate(LABEL_ORDER)}
IDX_TO_LABEL = {i: label for label, i in LABEL_TO_IDX.items()}

print("TF:", tf.__version__)
print("NumPy:", np.__version__)
print("Mixed precision:", mixed_precision.global_policy().name)
print("Artifacts:", ART)


TF: 2.20.0
NumPy: 2.4.1
Mixed precision: float32
Artifacts: d:\study apply\ML Ops\MasterArbeit_MLops\notebooks\model_HAR_27may_artifacts


In [4]:
# ============================================================
# Cell 2 ? Helpers: labels, clean windowing, augmentation, metrics
# ============================================================

def add_label_index(df):
    if "Activity" in df.columns and "activity" not in df.columns:
        df = df.rename(columns={"Activity": "activity"})
    df = df.copy()
    df["label_idx"] = df["activity"].map(LABEL_TO_IDX)
    df = df.dropna(subset=["label_idx"]).reset_index(drop=True)
    df["label_idx"] = df["label_idx"].astype(np.int32)
    return df


def check_required_columns(df, name):
    required = SENSOR_COLS + ["activity", "User"]
    missing = [c for c in required if c not in df.columns]
    assert not missing, f"{name} missing columns: {missing}"


def make_class_weights(y):
    present = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=present, y=y)
    out = {i: 1.0 for i in range(N_CLASSES)}
    for cls, w in zip(present, weights):
        out[int(cls)] = float(w)
    return out


def window_by_user_activity(df, sensor_cols=SENSOR_COLS, window=WINDOW, stride=STRIDE):
    """Clean User x contiguous-activity windowing.

    Windows never cross activity boundaries. This avoids mixed labels at transitions.
    Output shape: (N, 200, 6).
    """
    X_list, y_list, user_list = [], [], []

    for user_id, df_user in df.groupby("User", sort=False):
        df_user = df_user.reset_index(drop=True)
        seg_id = (df_user["activity"] != df_user["activity"].shift()).cumsum()

        for _, df_seg in df_user.groupby(seg_id, sort=False):
            if len(df_seg) < window:
                continue
            df_seg = df_seg.reset_index(drop=True)
            X_seg = df_seg[sensor_cols].values.astype(np.float32)
            y_seg = df_seg["label_idx"].values.astype(np.int32)

            for start in range(0, len(df_seg) - window + 1, stride):
                end = start + window
                assert len(np.unique(y_seg[start:end])) == 1, "Mixed-label segment"
                X_list.append(X_seg[start:end])
                y_list.append(int(y_seg[start]))
                user_list.append(int(user_id))

    return (
        np.array(X_list, dtype=np.float32),
        np.array(y_list, dtype=np.int32),
        np.array(user_list, dtype=np.int32),
    )


def print_window_summary(name, X, y):
    print(f"\n{name}")
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("class distribution:", np.bincount(y, minlength=N_CLASSES))


def aug_jitter(x, sigma=0.03):
    return x + np.random.normal(0, sigma, size=x.shape).astype(np.float32)


def aug_magnitude(x, sigma=0.10):
    scale = np.random.normal(1.0, sigma, size=(1, x.shape[1])).astype(np.float32)
    return x * scale


def aug_time_warp(x, sigma=0.20):
    n = x.shape[0]
    random_curve = np.random.normal(1.0, sigma, size=n)
    new_idx = np.cumsum(random_curve)
    new_idx = (new_idx - new_idx.min()) / (new_idx.max() - new_idx.min()) * (n - 1)
    warped = np.zeros_like(x)
    for ch in range(x.shape[1]):
        warped[:, ch] = np.interp(np.arange(n), new_idx, x[:, ch])
    return warped.astype(np.float32)


def augment_one(x):
    if np.random.rand() < 0.60:
        x = aug_time_warp(x)
    if np.random.rand() < 0.70:
        x = aug_magnitude(x)
    if np.random.rand() < 0.50:
        x = aug_jitter(x)
    return x.astype(np.float32)


def augment_array(X, multiplier=3):
    aug_Xs = [X]
    for _ in range(multiplier - 1):
        batch = np.empty_like(X)
        for i in range(len(X)):
            batch[i] = augment_one(X[i].copy())
        aug_Xs.append(batch)
    return np.concatenate(aug_Xs, axis=0).astype(np.float32)


def bootstrap_accuracy_ci(y_true, y_pred, n_boot=1000, seed=SEED):
    rng = np.random.RandomState(seed)
    vals = []
    for _ in range(n_boot):
        idx = rng.choice(len(y_true), len(y_true), replace=True)
        vals.append(accuracy_score(y_true[idx], y_pred[idx]))
    return [float(x) for x in np.percentile(vals, [2.5, 97.5])]


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")


def md5_file(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


In [6]:
# ============================================================
# Cell 3 — Load ADAMSense/anxiety_dataset source corpus
# ============================================================

ANX_PATH = ROOT / "data" / "anxiety_dataset.csv"
if not ANX_PATH.exists():
    ANX_PATH = ROOT / "_archive" / "6th may" / "data" / "anxiety_dataset.csv"
assert ANX_PATH.exists(), f"anxiety_dataset.csv not found: {ANX_PATH}"

df_pre = pd.read_csv(
    ANX_PATH,
    usecols=[
        "Time_w", "Ax_w", "Ay_w", "Az_w", "Gx_w", "Gy_w", "Gz_w",
        "Activity", "User",
    ],
)
df_pre = df_pre.rename(columns={"Time_w": "timestamp", "Activity": "activity"})
check_required_columns(df_pre, "df_pre")
df_pre = add_label_index(df_pre)

# Convert ADAMSense accelerometer from g to m/s^2 so the source domain matches
# Garmin's physical scale (Garmin pipeline already converts milliG -> m/s^2).
# Without this, the ADAMSense scaler fit on g-range values (~2) produces huge
# z-scores when reused on Garmin m/s^2 inputs in the Cell 8 zero-shot check,
# and the pretrained CNN learns features at a scale Garmin fine-tuning has to
# unlearn.
G_TO_MS2 = 9.80665
ACC_COLS = ["Ax_w", "Ay_w", "Az_w"]
accel_max_before = float(df_pre[ACC_COLS].abs().max().max())
df_pre[ACC_COLS] = df_pre[ACC_COLS] * G_TO_MS2
accel_max_after = float(df_pre[ACC_COLS].abs().max().max())

print("ADAMSense/anxiety shape:", df_pre.shape)
print("Users:", sorted(df_pre["User"].unique().astype(int).tolist()))
print(f"Accel max abs before g->m/s^2: {accel_max_before:.4f}  (expect ~2 if raw g)")
print(f"Accel max abs after  g->m/s^2: {accel_max_after:.4f}  (expect ~20)")
print(df_pre["activity"].value_counts())


ADAMSense/anxiety shape: (709582, 10)
Users: [2, 3, 5, 6, 7, 8, 10, 14, 15, 16]
Accel max abs before g->m/s^2: 2.0000  (expect ~2 if raw g)
Accel max abs after  g->m/s^2: 19.6133  (expect ~20)
activity
nape_rubbing         67859
knuckles_cracking    67498
smoking              66937
ear_rubbing          66599
hair_pulling         65973
forehead_rubbing     64604
hand_scratching      64490
nail_biting          63457
hand_tapping         63220
sitting              59996
standing             58949
Name: count, dtype: int64


In [7]:
# ============================================================
# Cell 4 ? Fit ADAMSense scaler + create ADAMSense windows
# ============================================================

scaler_pre = StandardScaler()
scaler_pre.fit(df_pre[SENSOR_COLS].values)

df_pre_scaled = df_pre.copy()
df_pre_scaled[SENSOR_COLS] = scaler_pre.transform(df_pre[SENSOR_COLS].values)

X_pre, y_pre, user_pre = window_by_user_activity(df_pre_scaled)
print_window_summary("ADAMSense pretrain windows", X_pre, y_pre)

joblib.dump(scaler_pre, ART / "pretrain_adamsense_scaler.joblib")



ADAMSense pretrain windows
X shape: (6914, 200, 6)
y shape: (6914,)
class distribution: [650 630 642 628 618 658 618 663 584 651 572]


['d:\\study apply\\ML Ops\\MasterArbeit_MLops\\notebooks\\model_HAR_27may_artifacts\\pretrain_adamsense_scaler.joblib']

In [10]:
# ============================================================
# Cell 5 ? Build fixed 1D-CNN + BiLSTM model
# ============================================================

def build_d1_model():
    inputs = layers.Input(shape=(WINDOW, len(SENSOR_COLS)))

    x = layers.Conv1D(64, kernel_size=7, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(128, kernel_size=5, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(256, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Bidirectional(layers.LSTM(128))(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(N_CLASSES, activation="softmax", dtype="float32")(x)
    return models.Model(inputs, outputs, name="model_HAR_1Dcnn_bilstm")

model_preview = build_d1_model()
model_preview.summary()
print("Parameter count:", model_preview.count_params())


Model: "model_HAR_1Dcnn_bilstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 200, 6)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_6 (Conv1D)               │ (None, 200, 64)        │         2,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 200, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 200, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 200, 128)       │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 200, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 100, 256)       │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 100, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 100, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 100, 256)       │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 11)             │         1,419 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 967,499 (3.69 MB)

 Trainable params: 966,347 (3.69 MB)

 Non-trainable params: 1,152 (4.50 KB)

Parameter count: 967499


In [12]:
# ============================================================
# Cell 6 ? Pretrain on ADAMSense + save pretrained base
# ============================================================

X_pre_train, X_pre_val, y_pre_train, y_pre_val = train_test_split(
    X_pre, y_pre, test_size=0.10, random_state=SEED, stratify=y_pre
)

model_pre = build_d1_model()
model_pre.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

pretrain_path = ART / "pretrained_d1_base_g.keras"
pretrain_callbacks = [
    callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=15, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(str(pretrain_path), monitor="val_accuracy", mode="max", save_best_only=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=8, min_lr=1e-7, verbose=1),
    callbacks.CSVLogger(str(ART / "pretrain_adamsense_log.csv")),
]

history_pre = model_pre.fit(
    X_pre_train, y_pre_train,
    validation_data=(X_pre_val, y_pre_val),
    epochs=80,
    batch_size=64,
    class_weight=make_class_weights(y_pre_train),
    callbacks=pretrain_callbacks,
    verbose=1,
    shuffle=True,
)

best_pre = float(max(history_pre.history["val_accuracy"]))
print("Best ADAMSense pretrain val_accuracy:", best_pre)
model_pre.save(pretrain_path)
write_json(ART / "pretrain_adamsense_summary.json", {"best_val_accuracy": best_pre, "model_path": str(pretrain_path)})


Epoch 1/80
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 349ms/step - accuracy: 0.3802 - loss: 1.9450
Epoch 1: val_accuracy improved from None to 0.33237, saving model to d:\study apply\ML Ops\MasterArbeit_MLops\notebooks\model_HAR_27may_artifacts\pretrained_d1_base_g.keras
98/98 ━━━━━━━━━━━━━━━━━━━━ 41s 377ms/step - accuracy: 0.4552 - loss: 1.6119 - val_accuracy: 0.3324 - val_loss: 1.8827 - learning_rate: 0.0010
Epoch 2/80
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 347ms/step - accuracy: 0.5502 - loss: 1.1872
Epoch 2: val_accuracy improved from 0.33237 to 0.48410, saving model to d:\study apply\ML Ops\MasterArbeit_MLops\notebooks\model_HAR_27may_artifacts\pretrained_d1_base_g.keras
98/98 ━━━━━━━━━━━━━━━━━━━━ 35s 359ms/step - accuracy: 0.5763 - loss: 1.1267 - val_accuracy: 0.4841 - val_loss: 1.4241 - learning_rate: 0.0010
Epoch 3/80
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 348ms/step - accuracy: 0.6453 - loss: 0.9723
Epoch 3: val_accuracy improved from 0.48410 to 0.68353, saving model to d:\study apply\ML Ops\MasterArbeit_ML

In [13]:
# ============================================================
# Cell 7 ? Load Garmin users 1-6 labelled data
# ============================================================

GARMIN_PATH = ROOT / "data" / "all_users_data_labeled.csv"
assert GARMIN_PATH.exists(), f"Garmin labelled data not found: {GARMIN_PATH}"

df_garmin = pd.read_csv(GARMIN_PATH)
df_garmin = add_label_index(df_garmin)
check_required_columns(df_garmin, "df_garmin")

users = sorted(df_garmin["User"].unique().astype(int).tolist())
print("Garmin users:", users)
assert users == [1, 2, 3, 4, 5, 6], "Expected Garmin users 1-6 only in all_users_data_labeled.csv"
assert 7 not in users, "User 7 must not be in training/selection data"
print("Rows:", len(df_garmin))
print(pd.crosstab(df_garmin["User"], df_garmin["activity"]))


Garmin users: [1, 2, 3, 4, 5, 6]
Rows: 385326
activity  ear_rubbing  forehead_rubbing  hair_pulling  hand_scratching  \
User                                                                     
1                5924              5714          6474             5524   
2                5975              5836          5739             5429   
3                5518              5672          5678             5949   
4                5684              5735          5720             5717   
5                5630              6075          5729             5974   
6                6037              6187          6121             6382   

activity  hand_tapping  knuckles_cracking  nail_biting  nape_rubbing  sitting  \
User                                                                            
1                 5029               5623         5967          5427     5765   
2                 5730               6182         8116          5672     6723   
3                 5278               

In [14]:
# ============================================================
# Cell 8 ? Zero-shot Garmin check before fine-tuning
# Uses ADAMSense/pretrain scaler. Diagnostic only.
# ============================================================

scaler_pre = joblib.load(ART / "pretrain_adamsense_scaler.joblib")
df_zero = df_garmin.iloc[:min(50000, len(df_garmin))].reset_index(drop=True).copy()
df_zero[SENSOR_COLS] = scaler_pre.transform(df_zero[SENSOR_COLS].values)
X_zero, y_zero, _ = window_by_user_activity(df_zero)
print_window_summary("Zero-shot Garmin windows", X_zero, y_zero)

model_zero = models.load_model(ART / "pretrained_d1_base_g.keras")
y_zero_pred = model_zero.predict(X_zero, verbose=0).argmax(axis=1)
zero_acc = float(accuracy_score(y_zero, y_zero_pred))
zero_f1 = float(f1_score(y_zero, y_zero_pred, average="macro", labels=np.arange(N_CLASSES), zero_division=0))
print("Zero-shot Garmin accuracy:", zero_acc)
print("Zero-shot Garmin macro-F1:", zero_f1)
write_json(ART / "zero_shot_garmin_summary.json", {"accuracy": zero_acc, "macro_f1": zero_f1})



Zero-shot Garmin windows
X shape: (488, 200, 6)
y shape: (488,)
class distribution: [58 56 63 54 49 55 58 53  0 42  0]
Zero-shot Garmin accuracy: 0.3094262295081967
Zero-shot Garmin macro-F1: 0.17927149588918134


In [15]:
# ============================================================
# Cell 9 ? Garmin adaptation sanity check: train users 1-5, validate user 6
# Not final model. Not official evaluation. Purpose: can pretrained model adapt to Garmin?
# ============================================================

TRAIN_USERS_SANITY = [1, 2, 3, 4, 5]
VAL_USERS_SANITY = [6]

df_train = df_garmin[df_garmin["User"].isin(TRAIN_USERS_SANITY)].reset_index(drop=True)
df_val = df_garmin[df_garmin["User"].isin(VAL_USERS_SANITY)].reset_index(drop=True)

scaler_sanity = StandardScaler()
scaler_sanity.fit(df_train[SENSOR_COLS].values)

df_train_s = df_train.copy()
df_val_s = df_val.copy()
df_train_s[SENSOR_COLS] = scaler_sanity.transform(df_train[SENSOR_COLS].values)
df_val_s[SENSOR_COLS] = scaler_sanity.transform(df_val[SENSOR_COLS].values)

X_tr, y_tr, _ = window_by_user_activity(df_train_s)
X_va, y_va, _ = window_by_user_activity(df_val_s)
print_window_summary("Sanity train windows users 1-5", X_tr, y_tr)
print_window_summary("Sanity val windows user 6", X_va, y_va)

X_tr_aug = augment_array(X_tr, multiplier=3)
y_tr_aug = np.concatenate([y_tr] * 3)

model_sanity = models.load_model(ART / "pretrained_d1_base_g.keras")

# Phase 1: train BiLSTM/head first
for layer in model_sanity.layers:
    if isinstance(layer, (layers.Conv1D, layers.BatchNormalization)):
        layer.trainable = False
model_sanity.compile(optimizer=optimizers.Adam(3e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_sanity.fit(
    X_tr_aug, y_tr_aug,
    validation_data=(X_va, y_va),
    epochs=80,
    batch_size=64,
    class_weight=make_class_weights(y_tr),
    callbacks=[callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=15, restore_best_weights=True, verbose=1)],
    verbose=1,
)

# Phase 2: low-LR full unfreeze
for layer in model_sanity.layers:
    layer.trainable = True
model_sanity.compile(optimizer=optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_sanity = model_sanity.fit(
    X_tr_aug, y_tr_aug,
    validation_data=(X_va, y_va),
    epochs=100,
    batch_size=64,
    class_weight=make_class_weights(y_tr),
    callbacks=[callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=25, restore_best_weights=True, verbose=1)],
    verbose=1,
)

y_va_pred = model_sanity.predict(X_va, verbose=0).argmax(axis=1)
sanity_acc = float(accuracy_score(y_va, y_va_pred))
sanity_f1 = float(f1_score(y_va, y_va_pred, average="macro", labels=np.arange(N_CLASSES), zero_division=0))
print("User 6 sanity accuracy:", sanity_acc)
print("User 6 sanity macro-F1:", sanity_f1)
model_sanity.save(ART / "model_HAR_1Dcnn_bilstm_user6_sanity.keras")
write_json(ART / "user6_sanity_summary.json", {"accuracy": sanity_acc, "macro_f1": sanity_f1})



Sanity train windows users 1-5
X shape: (3097, 200, 6)
y shape: (3097,)
class distribution: [280 283 286 279 266 281 323 260 285 272 282]

Sanity val windows user 6
X shape: (658, 200, 6)
y shape: (658,)
class distribution: [59 60 60 62 59 58 58 60 55 80 47]
Epoch 1/80
146/146 ━━━━━━━━━━━━━━━━━━━━ 44s 283ms/step - accuracy: 0.4897 - loss: 2.4422 - val_accuracy: 0.6064 - val_loss: 1.2909
Epoch 2/80
146/146 ━━━━━━━━━━━━━━━━━━━━ 40s 272ms/step - accuracy: 0.7667 - loss: 0.7128 - val_accuracy: 0.6033 - val_loss: 1.2350
Epoch 3/80
146/146 ━━━━━━━━━━━━━━━━━━━━ 40s 271ms/step - accuracy: 0.8726 - loss: 0.4027 - val_accuracy: 0.6505 - val_loss: 1.2342
Epoch 4/80
146/146 ━━━━━━━━━━━━━━━━━━━━ 39s 269ms/step - accuracy: 0.9243 - loss: 0.2518 - val_accuracy: 0.6489 - val_loss: 1.3948
Epoch 5/80
146/146 ━━━━━━━━━━━━━━━━━━━━ 39s 265ms/step - accuracy: 0.9492 - loss: 0.1664 - val_accuracy: 0.6322 - val_loss: 1.5799
Epoch 6/80
146/146 ━━━━━━━━━━━━━━━━━━━━ 39s 264ms/step - accuracy: 0.9676 - loss: 0.1

In [16]:
# ============================================================
# Cell 10a — Mentor-comparable 5-fold window-level CV on Garmin users 1-6
# Clearly labelled within-distribution learnability, not unseen-user generalization.
# ============================================================

scaler_cv = StandardScaler()
scaler_cv.fit(df_garmin[SENSOR_COLS].values)

df_cv = df_garmin.copy()
df_cv[SENSOR_COLS] = scaler_cv.transform(df_garmin[SENSOR_COLS].values)
X_cv, y_cv, users_cv = window_by_user_activity(df_cv)
print_window_summary("5-fold CV windows", X_cv, y_cv)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cv, y_cv), start=1):
    print(f"\n=== 5-fold CV fold {fold}/5 ===")
    Xtr, ytr = X_cv[tr_idx], y_cv[tr_idx]
    Xva, yva = X_cv[va_idx], y_cv[va_idx]
    Xtr_aug = augment_array(Xtr, multiplier=3)
    ytr_aug = np.concatenate([ytr] * 3)

    m = models.load_model(ART / "pretrained_d1_base_g.keras")
    for layer in m.layers:
        if isinstance(layer, (layers.Conv1D, layers.BatchNormalization)):
            layer.trainable = False
    m.compile(optimizer=optimizers.Adam(3e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    m.fit(Xtr_aug, ytr_aug, validation_data=(Xva, yva), epochs=60, batch_size=64,
          class_weight=make_class_weights(ytr), callbacks=[callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=10, restore_best_weights=True, verbose=0)], verbose=0)

    for layer in m.layers:
        layer.trainable = True
    m.compile(optimizer=optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    m.fit(Xtr_aug, ytr_aug, validation_data=(Xva, yva), epochs=80, batch_size=64,
          class_weight=make_class_weights(ytr), callbacks=[callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=15, restore_best_weights=True, verbose=0)], verbose=0)

    pred = m.predict(Xva, verbose=0).argmax(axis=1)
    acc = float(accuracy_score(yva, pred))
    f1 = float(f1_score(yva, pred, average="macro", labels=np.arange(N_CLASSES), zero_division=0))
    cv_results.append({"fold": fold, "accuracy": acc, "macro_f1": f1, "n_windows": int(len(yva))})
    print(f"Fold {fold}: accuracy={acc:.4f}, macro-F1={f1:.4f}")

cv_df = pd.DataFrame(cv_results)
cv_summary = {
    "protocol": "mentor_comparable_window_level_5fold_cv_on_garmin_users_1_6",
    "notes": "Scaler fitted globally on users 1-6 before splitting; result is within-distribution learnability and is optimistic relative to LOUO. Use LOUO for unseen-user generalization claims.",
    "folds": cv_results,
    "mean_accuracy": float(cv_df["accuracy"].mean()),
    "std_accuracy": float(cv_df["accuracy"].std()),
    "mean_macro_f1": float(cv_df["macro_f1"].mean()),
    "std_macro_f1": float(cv_df["macro_f1"].std()),
}
print(cv_df)
print("Mean accuracy:", cv_summary["mean_accuracy"])
print("Mean macro-F1:", cv_summary["mean_macro_f1"])
write_json(ART / "garmin_5fold_window_cv_results.json", cv_summary)



5-fold CV windows
X shape: (3755, 200, 6)
y shape: (3755,)
class distribution: [339 343 346 341 325 339 381 320 340 352 329]

=== 5-fold CV fold 1/5 ===
Fold 1: accuracy=0.9334, macro-F1=0.9342

=== 5-fold CV fold 2/5 ===
Fold 2: accuracy=0.9441, macro-F1=0.9447

=== 5-fold CV fold 3/5 ===
Fold 3: accuracy=0.9414, macro-F1=0.9422

=== 5-fold CV fold 4/5 ===
Fold 4: accuracy=0.9348, macro-F1=0.9352

=== 5-fold CV fold 5/5 ===
Fold 5: accuracy=0.9268, macro-F1=0.9278
   fold  accuracy  macro_f1  n_windows
0     1  0.933422  0.934203        751
1     2  0.944075  0.944726        751
2     3  0.941411  0.942198        751
3     4  0.934754  0.935248        751
4     5  0.926764  0.927769        751
Mean accuracy: 0.9360852197070573
Mean macro-F1: 0.9368288128397804


In [17]:
# ============================================================
# Cell 10b — LOUO across Garmin users 1-6 (strict)
# Held-out user is NEVER used for early stopping or model selection.
# Early stopping uses an inner 10% stratified split of the training users only.
# ============================================================

LOUO_USERS = [1, 2, 3, 4, 5, 6]
louo_results = []

for held_user in LOUO_USERS:
    print(f"\n=== LOUO held-out user {held_user} ===")
    np.random.seed(SEED + held_user)
    tf.random.set_seed(SEED + held_user)

    train_fold = df_garmin[df_garmin["User"] != held_user].reset_index(drop=True)
    test_fold = df_garmin[df_garmin["User"] == held_user].reset_index(drop=True)

    scaler_fold = StandardScaler()
    scaler_fold.fit(train_fold[SENSOR_COLS].values)

    train_s = train_fold.copy()
    test_s = test_fold.copy()
    train_s[SENSOR_COLS] = scaler_fold.transform(train_fold[SENSOR_COLS].values)
    test_s[SENSOR_COLS] = scaler_fold.transform(test_fold[SENSOR_COLS].values)

    Xtr_all, ytr_all, _ = window_by_user_activity(train_s)
    Xte, yte, _ = window_by_user_activity(test_s)

    # Inner train/val split: training users only. The held-out user's windows
    # (Xte, yte) are touched exactly once, after training, for the final score.
    Xtr_in, Xval_in, ytr_in, yval_in = train_test_split(
        Xtr_all, ytr_all,
        test_size=0.10,
        random_state=SEED + held_user,
        stratify=ytr_all,
    )

    Xtr_aug = augment_array(Xtr_in, multiplier=3)
    ytr_aug = np.concatenate([ytr_in] * 3)

    m = models.load_model(ART / "pretrained_d1_base_g.keras")

    for layer in m.layers:
        if isinstance(layer, (layers.Conv1D, layers.BatchNormalization)):
            layer.trainable = False
    m.compile(optimizer=optimizers.Adam(3e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    m.fit(Xtr_aug, ytr_aug, validation_data=(Xval_in, yval_in), epochs=60, batch_size=128,
          class_weight=make_class_weights(ytr_in),
          callbacks=[callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=15, restore_best_weights=True, verbose=0)], verbose=0)

    for layer in m.layers:
        layer.trainable = True
    m.compile(optimizer=optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    m.fit(Xtr_aug, ytr_aug, validation_data=(Xval_in, yval_in), epochs=100, batch_size=128,
          class_weight=make_class_weights(ytr_in),
          callbacks=[callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=25, restore_best_weights=True, verbose=0)], verbose=0)

    # Final single-shot evaluation on the held-out user.
    pred = m.predict(Xte, verbose=0).argmax(axis=1)
    acc = float(accuracy_score(yte, pred))
    f1 = float(f1_score(yte, pred, average="macro", labels=np.arange(N_CLASSES), zero_division=0))
    louo_results.append({"held_user": held_user, "accuracy": acc, "macro_f1": f1, "n_windows": int(len(yte))})
    print(f"User {held_user}: accuracy={acc:.4f}, macro-F1={f1:.4f}, n={len(yte)}")

louo_df = pd.DataFrame(louo_results)
louo_summary = {
    "protocol": "LOUO_users_1_6_train_only_scaler_per_fold",
    "early_stopping_signal": "inner 10% stratified split of training users; held-out user never used for selection",
    "folds": louo_results,
    "mean_accuracy": float(louo_df["accuracy"].mean()),
    "std_accuracy": float(louo_df["accuracy"].std()),
    "mean_macro_f1": float(louo_df["macro_f1"].mean()),
    "std_macro_f1": float(louo_df["macro_f1"].std()),
}
print("\nLOUO summary")
print(louo_df)
print("Mean accuracy:", louo_summary["mean_accuracy"])
print("Mean macro-F1:", louo_summary["mean_macro_f1"])
write_json(ART / "garmin_louo_users_1_6_results.json", louo_summary)



=== LOUO held-out user 1 ===
User 1: accuracy=0.4552, macro-F1=0.4176, n=591

=== LOUO held-out user 2 ===
User 2: accuracy=0.0569, macro-F1=0.0572, n=668

=== LOUO held-out user 3 ===
User 3: accuracy=0.5050, macro-F1=0.4743, n=600

=== LOUO held-out user 4 ===
User 4: accuracy=0.5375, macro-F1=0.5186, n=614

=== LOUO held-out user 5 ===
User 5: accuracy=0.5000, macro-F1=0.4410, n=624

=== LOUO held-out user 6 ===
User 6: accuracy=0.6109, macro-F1=0.6038, n=658

LOUO summary
   held_user  accuracy  macro_f1  n_windows
0          1  0.455161  0.417648        591
1          2  0.056886  0.057244        668
2          3  0.505000  0.474331        600
3          4  0.537459  0.518622        614
4          5  0.500000  0.440972        624
5          6  0.610942  0.603791        658
Mean accuracy: 0.4442414174455833
Mean macro-F1: 0.4187681119617374


In [19]:
# ============================================================
# Cell 11 — Final freeze training on all Garmin users 1-6 + save runtime contract
# Validation split is taken BEFORE augmentation so the val set never contains
# augmented twins of training windows. Validation is a within-distribution
# random window split, so val accuracy is optimistic; the honest cross-user
# number is Cell 10b's LOUO.
# ============================================================

# Final train-only scaler on all Garmin users 1-6
scaler_final = StandardScaler()
scaler_final.fit(df_garmin[SENSOR_COLS].values)

df_final = df_garmin.copy()
df_final[SENSOR_COLS] = scaler_final.transform(df_garmin[SENSOR_COLS].values)
X_final, y_final, user_final = window_by_user_activity(df_final)
print_window_summary("Final users 1-6 training windows", X_final, y_final)

# Pre-augmentation split so validation never contains augmented copies of
# training windows. The early-stopping signal is therefore clean.
X_final_train, X_final_val, y_final_train, y_final_val = train_test_split(
    X_final, y_final,
    test_size=0.10,
    random_state=SEED,
    stratify=y_final,
)
X_final_train_aug = augment_array(X_final_train, multiplier=3)
y_final_train_aug = np.concatenate([y_final_train] * 3)
print("Final train (pre-aug):", X_final_train.shape, "val:", X_final_val.shape, "train (post-aug):", X_final_train_aug.shape)

model_final = models.load_model(ART / "pretrained_d1_base_g.keras")

# Phase 1: BiLSTM/head adaptation
for layer in model_final.layers:
    if isinstance(layer, (layers.Conv1D, layers.BatchNormalization)):
        layer.trainable = False
model_final.compile(optimizer=optimizers.Adam(3e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_final.fit(X_final_train_aug, y_final_train_aug,
                validation_data=(X_final_val, y_final_val),
                epochs=80, batch_size=64,
                class_weight=make_class_weights(y_final_train),
                callbacks=[callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=15, restore_best_weights=True, verbose=1)],
                verbose=1, shuffle=True)

# Phase 2: full low-LR polish
for layer in model_final.layers:
    layer.trainable = True
model_final.compile(optimizer=optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_final.fit(X_final_train_aug, y_final_train_aug,
                validation_data=(X_final_val, y_final_val),
                epochs=100, batch_size=64,
                class_weight=make_class_weights(y_final_train),
                callbacks=[callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=25, restore_best_weights=True, verbose=1)],
                verbose=1, shuffle=True)

final_train_acc = float(model_final.evaluate(X_final_train, y_final_train, verbose=0)[1])
final_val_acc = float(model_final.evaluate(X_final_val, y_final_val, verbose=0)[1])
print(f"\nFinal model — within-distribution numbers (NOT cross-user generalization):")
print(f"  train (clean, no aug) accuracy: {final_train_acc:.4f}")
print(f"  val   (10% random window split) accuracy: {final_val_acc:.4f}")
print("Compare against Cell 10b LOUO mean before treating this model as deployable.")

# ---------------------------------------------------------------
# Runtime contract artifact saves
# ---------------------------------------------------------------
FINAL_ID = f"model_d1_final_{datetime.now().strftime('%Y%m%d')}"
RUNTIME_DIR = ROOT / "models" / "runtime_contract" / FINAL_ID
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ARTIFACT_NAME = "finetuned_1DCNNBILSTM_HAR_model"
model_path = RUNTIME_DIR / f"{MODEL_ARTIFACT_NAME}.keras"
model_final.save(model_path)

# Also save a TF SavedModel directory for serving/TFSMLayer compatibility.
savedmodel_dir = RUNTIME_DIR / f"{MODEL_ARTIFACT_NAME}_savedmodel"
if hasattr(model_final, "export"):
    model_final.export(str(savedmodel_dir))
else:
    model_final.save(str(savedmodel_dir), save_format="tf")
print("Saved native .keras:", model_path)
print("Saved SavedModel  :", savedmodel_dir)


# Training columns carry the ADAMSense `_w` wrist suffix; the production
# pipeline / API expects canonical names without it. Contract exposes canonical
# names and ships an alias map so the runtime knows which Garmin column to
# feed into each model channel.
CANONICAL_COLS = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]
TRAINING_COLUMN_ALIASES = dict(zip(CANONICAL_COLS, SENSOR_COLS))

scaler_json = {
    "feature_order": CANONICAL_COLS,
    "training_feature_order": SENSOR_COLS,
    "training_column_aliases": TRAINING_COLUMN_ALIASES,
    "mean": scaler_final.mean_.tolist(),
    "scale": scaler_final.scale_.tolist(),
    "var": scaler_final.var_.tolist(),
    "scaler_mean": scaler_final.mean_.tolist(),
    "scaler_scale": scaler_final.scale_.tolist(),
    "scaler_var": scaler_final.var_.tolist(),
    "scaler_type": "StandardScaler",
    "fit_scope": "Garmin users 1-6 only; User 7 excluded",
    "fit_rule": "Apply after unit conversion to m/s^2. Never fit on inference data.",
}
write_json(RUNTIME_DIR / "garmin_train_scaler.json", scaler_json)
joblib.dump(scaler_final, RUNTIME_DIR / "garmin_train_scaler.joblib")

write_json(RUNTIME_DIR / "label_order.json", {"label_order": LABEL_ORDER, "label_to_idx": LABEL_TO_IDX, "idx_to_label": IDX_TO_LABEL})

baseline = {
    "feature_order": CANONICAL_COLS,
    "mean": X_final.mean(axis=(0, 1)).tolist(),
    "std": X_final.std(axis=(0, 1)).tolist(),
    "source": "Garmin users 1-6 scaled with final users 1-6 scaler",
}
write_json(RUNTIME_DIR / "garmin_train_scaled_baseline.json", baseline)

louo_path = ART / "garmin_louo_users_1_6_results.json"
cv_path = ART / "garmin_5fold_window_cv_results.json"
contract = {
    "schema_version": 1,
    "model_id": FINAL_ID,
    "model_family": "D1 1D-CNN + BiLSTM, ADAMSense pretrain (g->m/s^2), Garmin users 1-6 final freeze",
    "input_shape": [WINDOW, len(SENSOR_COLS)],
    "output_classes": N_CLASSES,
    "feature_order": CANONICAL_COLS,
    "training_column_aliases": TRAINING_COLUMN_ALIASES,
    "label_order": LABEL_ORDER,
    "window_size": WINDOW,
    "stride": STRIDE,
    "expected_hz": 50,
    "accelerometer_unit": "m/s^2",
    "gyroscope_unit": "deg/s",
    "final_scaler_fit_users": [1, 2, 3, 4, 5, 6],
    "user7_excluded_from_training": True,
    "final_freeze_validation_protocol": "pre-augmentation 10% stratified split; augmentation applied only to training partition",
    "within_distribution_val_accuracy": final_val_acc,
    "pretrain_path": str(ANX_PATH.relative_to(ROOT)).replace("\\", "/"),
    "garmin_train_path": str(GARMIN_PATH.relative_to(ROOT)).replace("\\", "/"),
    "louo_results_file": str(louo_path.relative_to(ROOT)).replace("\\", "/") if louo_path.exists() else None,
    "fivefold_results_file": str(cv_path.relative_to(ROOT)).replace("\\", "/") if cv_path.exists() else None,
    "discipline_statement": "No architecture, hyperparameter, augmentation, epoch, threshold, or acceptance decision was changed after observing User 7 results.",
}
write_json(RUNTIME_DIR / "model_contract.json", contract)

md5s = {p.name: md5_file(p) for p in sorted(RUNTIME_DIR.iterdir()) if p.is_file()}
write_json(RUNTIME_DIR / "artifact_md5.json", md5s)

print("\nRuntime contract saved:", RUNTIME_DIR)
print(json.dumps(md5s, indent=2))



Final users 1-6 training windows
X shape: (3755, 200, 6)
y shape: (3755,)
class distribution: [339 343 346 341 325 339 381 320 340 352 329]
Final train (pre-aug): (3379, 200, 6) val: (376, 200, 6) train (post-aug): (10137, 200, 6)
Epoch 1/80
159/159 ━━━━━━━━━━━━━━━━━━━━ 134s 805ms/step - accuracy: 0.5089 - loss: 2.3575 - val_accuracy: 0.7207 - val_loss: 0.8851
Epoch 2/80
159/159 ━━━━━━━━━━━━━━━━━━━━ 115s 724ms/step - accuracy: 0.7632 - loss: 0.7405 - val_accuracy: 0.8112 - val_loss: 0.5758
Epoch 3/80
159/159 ━━━━━━━━━━━━━━━━━━━━ 119s 747ms/step - accuracy: 0.8577 - loss: 0.4483 - val_accuracy: 0.8777 - val_loss: 0.4174
Epoch 4/80
159/159 ━━━━━━━━━━━━━━━━━━━━ 112s 704ms/step - accuracy: 0.9123 - loss: 0.2863 - val_accuracy: 0.9016 - val_loss: 0.3532
Epoch 5/80
159/159 ━━━━━━━━━━━━━━━━━━━━ 109s 684ms/step - accuracy: 0.9411 - loss: 0.1943 - val_accuracy: 0.9069 - val_loss: 0.3225
Epoch 6/80
159/159 ━━━━━━━━━━━━━━━━━━━━ 110s 694ms/step - accuracy: 0.9580 - loss: 0.1400 - val_accuracy: 0.

INFO:tensorflow:Assets written to: d:\study apply\ML Ops\MasterArbeit_MLops\models\runtime_contract\model_d1_final_20260528\finetuned_1DCNNBILSTM_HAR_model_savedmodel\assets


Saved artifact at 'd:\study apply\ML Ops\MasterArbeit_MLops\models\runtime_contract\model_d1_final_20260528\finetuned_1DCNNBILSTM_HAR_model_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 200, 6), dtype=tf.float32, name='input_layer_4')
Output Type:
  TensorSpec(shape=(None, 11), dtype=tf.float32, name=None)
Captures:
  2761878249616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2761878250384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2761878251536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2761878252496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2761878252304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2761878251728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2761878252880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2761878250960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2761878253072: TensorSpec(shape=(), dtype=tf.resource, nam

In [20]:
# ============================================================
# Cell 12 — User 7 single-shot blind probe
# Run only after Cell 11 freezes the final users-1-6 model and scaler.
# Descriptive only: no model choices may be changed after this.
# ============================================================

# Canonical labeled User 7 file per data/labels/2026-05-05-15-11-15/LABELING_REPORT.md
# ("Ready for evaluation/analysis using the corrected `_ms2.csv` labeled file").
# Accel is in m/s^2; gyro in deg/s; activities match the official LABEL_ORDER.
USER7_PATH = ROOT / "data" / "labels" / "2026-05-05-15-11-15" / "2026-05-05-15-11-15_labeled_50Hz_user7_ms2.csv"
assert USER7_PATH.exists(), f"User 7 file not found: {USER7_PATH}"
print("Using User 7 file:", USER7_PATH)

df_u7 = pd.read_csv(USER7_PATH)
df_u7 = add_label_index(df_u7)
check_required_columns(df_u7, "df_u7")

# Load latest runtime contract from Cell 11. NOTE: Cell 11 currently has its
# artifact saves DISABLED. Re-enable the save block in Cell 11 before running
# this cell, otherwise no runtime_contract folder will exist to load from.
try:
    runtime_dir = RUNTIME_DIR
except NameError:
    runtime_dirs = sorted((ROOT / "models" / "runtime_contract").glob("model_d1_final_*"))
    assert runtime_dirs, (
        "No model_d1_final runtime contract found. Re-enable the saves in "
        "Cell 11 and rerun it before running Cell 12."
    )
    runtime_dir = runtime_dirs[-1]

model_u7 = models.load_model(runtime_dir / "finetuned_1DCNNBILSTM_HAR_model.keras")
scaler_u7 = joblib.load(runtime_dir / "garmin_train_scaler.joblib")

df_u7_s = df_u7.copy()
df_u7_s[SENSOR_COLS] = scaler_u7.transform(df_u7[SENSOR_COLS].values)
X_u7, y_u7, _ = window_by_user_activity(df_u7_s)
print_window_summary("User 7 windows", X_u7, y_u7)

probs = model_u7.predict(X_u7, verbose=0)
y_pred = probs.argmax(axis=1)

u7_acc = float(accuracy_score(y_u7, y_pred))
u7_f1 = float(f1_score(y_u7, y_pred, average="macro", labels=np.arange(N_CLASSES), zero_division=0))
u7_conf = float(probs.max(axis=1).mean())
ci = bootstrap_accuracy_ci(y_u7, y_pred)

print("User 7 accuracy:", u7_acc)
print("User 7 macro-F1:", u7_f1)
print("User 7 mean confidence:", u7_conf)
print("User 7 95% bootstrap CI:", ci)
print(classification_report(
    y_u7, y_pred,
    labels=np.arange(N_CLASSES),
    target_names=LABEL_ORDER,
    zero_division=0,
))

user7_summary = {
    "user7_path": str(Path(USER7_PATH).relative_to(ROOT)).replace("\\", "/") if Path(USER7_PATH).is_relative_to(ROOT) else str(USER7_PATH),
    "accuracy": u7_acc,
    "macro_f1": u7_f1,
    "mean_confidence": u7_conf,
    "bootstrap_ci_95": ci,
    "note": "Descriptive blind probe only; not used for model selection or acceptance.",
}
write_json(runtime_dir / "user7_blind_probe_summary.json", user7_summary)
print("Saved:", runtime_dir / "user7_blind_probe_summary.json")


Using User 7 file: d:\study apply\ML Ops\MasterArbeit_MLops\data\labels\2026-05-05-15-11-15\2026-05-05-15-11-15_labeled_50Hz_user7_ms2.csv

User 7 windows
X shape: (974, 200, 6)
y shape: (974,)
class distribution: [88 85 90 90 88 88 89 91 88 88 89]
User 7 accuracy: 0.36960985626283366
User 7 macro-F1: 0.34964828143200904
User 7 mean confidence: 0.857867956161499
User 7 95% bootstrap CI: [0.3387833675564682, 0.39835728952772076]
                   precision    recall  f1-score   support

      ear_rubbing       0.40      0.49      0.44        88
 forehead_rubbing       0.39      0.34      0.36        85
     hair_pulling       0.00      0.00      0.00        90
  hand_scratching       0.13      0.12      0.13        90
     hand_tapping       0.51      0.94      0.66        88
knuckles_cracking       0.41      0.66      0.51        88
      nail_biting       0.27      0.18      0.21        89
     nape_rubbing       0.20      0.24      0.22        91
          sitting       0.98      0.

In [21]:
# ============================================================
# Cell 13 — Unlabeled User 7 inference probe
# ============================================================
# Same recording session as the labeled User 7 file but with no `activity` column.
# Two views:
#   (1) Per-activity prediction counts + mean confidence on all windows.
#   (2) Time-aligned comparison: for each known time segment from the labeling
#       report (which describes the same recording), what did the model predict?
# This is a deployment-style probe, not a thesis acceptance metric.

from collections import Counter

UNLABELED_USER7_PATH = ROOT / "data" / "raw" / "2026-05-05-15-11-15_unlabeled_50Hz_user7.csv"
assert UNLABELED_USER7_PATH.exists(), f"Unlabeled User 7 not found: {UNLABELED_USER7_PATH}"

# Find the most recent runtime contract from Cell 11
runtime_dirs = sorted((ROOT / "models" / "runtime_contract").glob("model_d1_final_*"))
assert runtime_dirs, "No model_d1_final runtime contract found. Run Cell 11 with saves enabled first."
runtime_dir = runtime_dirs[-1]
print("Using contract:", runtime_dir.name)

model_unl = models.load_model(runtime_dir / "finetuned_1DCNNBILSTM_HAR_model.keras")
scaler_unl = joblib.load(runtime_dir / "garmin_train_scaler.joblib")

df_unl = pd.read_csv(UNLABELED_USER7_PATH)
df_unl["timestamp"] = pd.to_datetime(df_unl["timestamp"])
t0 = df_unl["timestamp"].iloc[0]
df_unl["t_sec"] = (df_unl["timestamp"] - t0).dt.total_seconds()

# Apply final-freeze scaler
df_unl_s = df_unl.copy()
df_unl_s[SENSOR_COLS] = scaler_unl.transform(df_unl[SENSOR_COLS].values)

# Unlabeled windowing: stride across rows, no activity grouping required
arr = df_unl_s[SENSOR_COLS].values.astype(np.float32)
t_arr = df_unl_s["t_sec"].values
window_starts = list(range(0, len(arr) - WINDOW + 1, STRIDE))
X_unl = np.array([arr[s:s + WINDOW] for s in window_starts], dtype=np.float32)
window_center_t = np.array([t_arr[s + WINDOW // 2] for s in window_starts])
print("Unlabeled windows:", X_unl.shape)

# Inference
probs_unl = model_unl.predict(X_unl, verbose=0)
y_pred_unl = probs_unl.argmax(axis=1)
conf_unl = probs_unl.max(axis=1)

# ---- View 1: Per-activity prediction counts + mean confidence ----
print("\nPer-activity prediction counts (predicted class):")
counts = Counter(y_pred_unl.tolist())
for cls_idx in sorted(counts.keys()):
    label = IDX_TO_LABEL[int(cls_idx)]
    n = counts[cls_idx]
    pct = 100.0 * n / len(y_pred_unl)
    mean_conf = float(conf_unl[y_pred_unl == cls_idx].mean())
    print(f"  {label:20s} n={n:5d} ({pct:5.1f}%)  mean_conf={mean_conf:.3f}")

print(f"\nOverall confidence: mean={conf_unl.mean():.3f}  median={np.median(conf_unl):.3f}  std={conf_unl.std():.3f}")
print(f"Low-confidence windows (<0.5): {int((conf_unl < 0.5).sum())} ({100.0 * (conf_unl < 0.5).mean():.1f}%)")

# ---- View 2: Time-aligned per-segment comparison ----
# Schedule from data/labels/2026-05-05-15-11-15/LABELING_REPORT.md
# (start_sec, end_sec, activity); transition gaps are excluded.
SEGMENTS = [
    ("ear_rubbing",        0.00,    180.00),
    ("forehead_rubbing",   185.00,  360.64),
    ("hair_pulling",       360.64,  546.22),
    ("hand_scratching",    546.22,  733.55),
    ("hand_tapping",       739.02,  919.84),
    ("knuckles_cracking",  925.23,  1106.76),
    ("nail_biting",        1117.35, 1299.96),
    ("nape_rubbing",       1310.08, 1497.07),
    ("sitting",            1508.14, 1690.64),
    ("smoking",            1701.74, 1883.93),
    ("standing",           1895.42, 2080.41),
]

print("\nTime-aligned per-segment comparison vs LABELING_REPORT.md:")
print(f"  {'true_activity':22s} {'n_win':>6s}  {'top1_pred':>22s} {'top1_pct':>8s}  {'correct_pct':>11s}  {'mean_conf':>9s}")
segment_summary = []
for true_act, t_start, t_end in SEGMENTS:
    in_seg = (window_center_t >= t_start) & (window_center_t < t_end)
    n_in = int(in_seg.sum())
    if n_in == 0:
        continue
    preds_in = y_pred_unl[in_seg]
    confs_in = conf_unl[in_seg]
    seg_counts = Counter(preds_in.tolist())
    top_cls, top_n = seg_counts.most_common(1)[0]
    top_label = IDX_TO_LABEL[int(top_cls)]
    correct_n = int((preds_in == LABEL_TO_IDX[true_act]).sum()) if true_act in LABEL_TO_IDX else 0
    print(f"  {true_act:22s} {n_in:6d}  {top_label:>22s} {100.0*top_n/n_in:7.1f}%  {100.0*correct_n/n_in:10.1f}%  {confs_in.mean():9.3f}")
    segment_summary.append({
        "true_activity": true_act,
        "t_start_sec": t_start,
        "t_end_sec": t_end,
        "n_windows": n_in,
        "top_predicted": top_label,
        "top_predicted_pct": float(100.0 * top_n / n_in),
        "correct_n": correct_n,
        "correct_pct": float(100.0 * correct_n / n_in),
        "mean_confidence": float(confs_in.mean()),
    })

correct_total = sum(s["correct_n"] for s in segment_summary)
n_assigned = sum(s["n_windows"] for s in segment_summary)
print(f"\nWindow-level accuracy vs schedule: {correct_total}/{n_assigned} = {100.0*correct_total/max(n_assigned,1):.1f}%")

unlabeled_summary = {
    "unlabeled_path": str(UNLABELED_USER7_PATH.relative_to(ROOT)).replace("\\", "/"),
    "model_contract": runtime_dir.name,
    "n_windows": int(len(y_pred_unl)),
    "per_activity_counts": {IDX_TO_LABEL[int(cls)]: int(c) for cls, c in counts.items()},
    "mean_confidence": float(conf_unl.mean()),
    "median_confidence": float(np.median(conf_unl)),
    "low_confidence_pct": float(100.0 * (conf_unl < 0.5).mean()),
    "schedule_window_accuracy_pct": float(100.0 * correct_total / max(n_assigned, 1)),
    "segments": segment_summary,
}
write_json(runtime_dir / "user7_unlabeled_probe_summary.json", unlabeled_summary)
print("\nSaved:", runtime_dir / "user7_unlabeled_probe_summary.json")


Using contract: model_d1_final_20260528
Unlabeled windows: (989, 200, 6)

Per-activity prediction counts (predicted class):
  ear_rubbing          n=   98 (  9.9%)  mean_conf=0.831
  forehead_rubbing     n=   82 (  8.3%)  mean_conf=0.847
  hair_pulling         n=   57 (  5.8%)  mean_conf=0.764
  hand_scratching      n=   98 (  9.9%)  mean_conf=0.741
  hand_tapping         n=  161 ( 16.3%)  mean_conf=0.923
  knuckles_cracking    n=  148 ( 15.0%)  mean_conf=0.825
  nail_biting          n=   61 (  6.2%)  mean_conf=0.822
  nape_rubbing         n=  104 ( 10.5%)  mean_conf=0.843
  sitting              n=   63 (  6.4%)  mean_conf=0.959
  smoking              n=   99 ( 10.0%)  mean_conf=0.875
  standing             n=   18 (  1.8%)  mean_conf=0.775

Overall confidence: mean=0.846  median=0.956  std=0.197
Low-confidence windows (<0.5): 92 (9.3%)

Time-aligned per-segment comparison vs LABELING_REPORT.md:
  true_activity           n_win               top1_pred top1_pct  correct_pct  mean_conf
  

In [22]:
from datetime import datetime

In [23]:
# Runtime contract artifact saves
FINAL_ID = f"model_d1_final_{datetime.now().strftime('%Y%m%d')}"
RUNTIME_DIR = ROOT / "models" / "runtime_contract" / FINAL_ID
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ARTIFACT_NAME = "finetuned_1DCNNBILSTM_HAR_model"

model_path = RUNTIME_DIR / f"{MODEL_ARTIFACT_NAME}.keras"
model_final.save(model_path)

# Also save a TF SavedModel directory for serving/TFSMLayer compatibility.
savedmodel_dir = RUNTIME_DIR / f"{MODEL_ARTIFACT_NAME}_savedmodel"

if hasattr(model_final, "export"):
    # Keras 3 / thesis-mlops
    model_final.export(str(savedmodel_dir))
else:
    # Older tf.keras / tf-gpu fallback
    model_final.save(str(savedmodel_dir), save_format="tf")

print("Saved native .keras:", model_path)
print("Saved SavedModel  :", savedmodel_dir)

INFO:tensorflow:Assets written to: d:\study apply\ML Ops\MasterArbeit_MLops\models\runtime_contract\model_d1_final_20260528\finetuned_1DCNNBILSTM_HAR_model_savedmodel\assets


INFO:tensorflow:Assets written to: d:\study apply\ML Ops\MasterArbeit_MLops\models\runtime_contract\model_d1_final_20260528\finetuned_1DCNNBILSTM_HAR_model_savedmodel\assets


Saved artifact at 'd:\study apply\ML Ops\MasterArbeit_MLops\models\runtime_contract\model_d1_final_20260528\finetuned_1DCNNBILSTM_HAR_model_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 200, 6), dtype=tf.float32, name='input_layer_4')
Output Type:
  TensorSpec(shape=(None, 11), dtype=tf.float32, name=None)
Captures:
  2761878249616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2761878250384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2761878251536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2761878252496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2761878252304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2761878251728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2761878252880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2761878250960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2761878253072: TensorSpec(shape=(), dtype=tf.resource, nam